
# Assignment 4

This is a template notebook for Assignment 4.


## Install dependencies and initialization

In [ ]:
# Compiles the detectron2 code on your machine (or google colab machine) using the latest torch and python available
!pip install pycocotools>=2.0.1
#!pip install pyyaml==5.1
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'


# If running it locally you can use pre-complied installations files from the link below (not compatable to python 3.10 and above)
# https://detectron2.readthedocs.io/en/latest/tutorials/install.html

In [ ]:
!pwd # shows current directory
!ls  # shows all files in this directory
!nvidia-smi # shows the specs and the current status of the allocated GPU

In [ ]:
# import some common libraries
# from google.colab.patches import cv2_imshow
from sklearn.metrics import jaccard_score
from PIL import Image, ImageDraw
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import datetime
import random
import json
import cv2
import csv
import os

# import some common pytorch utilities
from torch.utils.data import Dataset, DataLoader, IterableDataset, default_collate
import torchvision.transforms as transforms
from torch.autograd import Variable
import torch.nn.functional as F
import torch.nn as nn
import torch



# import some common detectron2 utilities
import detectron2
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.structures import BoxMode
from detectron2.engine import DefaultTrainer
from detectron2.engine import DefaultPredictor
from detectron2.utils.logger import setup_logger
from detectron2.utils.visualizer import ColorMode
from detectron2.utils.visualizer import Visualizer
from detectron2.data import build_detection_test_loader
from detectron2.data import build_detection_train_loader
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.utils.events import TensorboardXWriter
from detectron2.config import CfgNode
from detectron2.data import detection_utils as utils
setup_logger()

In [ ]:
# Make sure that GPU is available for your notebook.
# Otherwise, you need to update the settungs in Runtime -> Change runtime type -> Hardware accelerator

torch.cuda.is_available()


In [ ]:
# Ignore it if running on a local machine and not on colab. 
# You need to mount your google drive in order to load the data:
# from google.colab import drive
# drive.mount('/content/drive')
# Put all the corresponding data files in a data folder and put the data folder in a same directory with this notebook.
# Also create an output directory for your files such as the trained models and the output images.

In [ ]:
# Define the location of current directory, which should contain data/train, data/test, and data/train.json.
# TODO: approx 1 line
BASE_DIR = './CMPT_CV_lab4'
OUTPUT_DIR = '{}/output'.format(BASE_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Part 1: Object Detection

### Data Loader

In [ ]:
'''
# This function should return a list of data samples in which each sample is a dictionary.
# Make sure to select the correct bbox_mode for the data
# For the test data, you only have access to the images, therefore, the annotations should be empty.
# Other values could be obtained from the image files.
# TODO: approx 35 lines
'''

def get_annotations(data_dirs):
    train_json_file = os.path.join(data_dirs, "train.json")
    with open(train_json_file) as f:
                annotations = json.load(f)
    return annotations

def get_train_data(data_dirs,annotations, condtion):
    dataset = []
    current_image_id = -1
    record = {}

    for entry in annotations:
        if condtion(entry['image_id']):
             
            image_id = entry['image_id']
            if image_id != current_image_id:
                if current_image_id != -1:
                    dataset.append(record)

                filename = os.path.join(data_dirs, "train", entry["file_name"])
                height, width = cv2.imread(filename).shape[:2]
                record = {
                    "file_name": filename,
                    "image_id": entry["image_id"],
                    "height": height,
                    "width": width,
                }
                current_image_id = image_id
                record["annotations"] = []

            # Add annotation to the current record
            obj = {
                "bbox": entry["bbox"],
                "bbox_mode": BoxMode.XYWH_ABS,
                "segmentation": entry["segmentation"],
                "category_id": entry["category_id"],
                "iscrowd": entry['iscrowd'],
            }
            record["annotations"].append(obj)

    # Append the last record
    if record:
        dataset.append(record)

    return dataset

def get_test_data(data_dirs):
    """Process test data without annotations."""
    dataset = []
    test_dir = os.path.join(data_dirs, "test")
    image_files = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

    for idx, img_file in enumerate(image_files):
        filename = os.path.join(test_dir, img_file)
        height, width = cv2.imread(filename).shape[:2]
        dataset.append({
            "file_name": filename,
            "image_id": idx,
            "height": height,
            "width": width,
            "annotations": [],
        })

    return dataset


def get_detection_data(set_name):
    data_dirs = '{}/data'.format(BASE_DIR)
    annos = get_annotations(data_dirs)
    if set_name == "train":
        return get_train_data(data_dirs, annos, lambda id: id <= 1000)
    if set_name == "validate":
        return get_train_data(data_dirs, annos, lambda id: id > 1000)
    if set_name == "test":
        return get_test_data(data_dirs)
    return []
    

In [ ]:
'''
# Remember to add your dataset to DatasetCatalog and MetadataCatalog
# Consdier "data_detection_train" and "data_detection_test" for registration
# You can also add an optional "data_detection_val" for your validation by spliting the training data
# TODO: approx 5 lines
'''
DatasetCatalog.clear()
MetadataCatalog.clear()
DatasetCatalog.register("data_detection_train", lambda d="train": get_detection_data(d))
DatasetCatalog.register("data_detection_val", lambda d="validate": get_detection_data(d))
DatasetCatalog.register("data_detection_test", lambda d="test": get_detection_data(d))
MetadataCatalog.get("data_detection_train").thing_classes = ["ship","storage tank","baseball diamond","tennis court","plane"] 
MetadataCatalog.get("data_detection_test").thing_classes = ["ship","storage tank","baseball diamond","tennis court","plane"] 
MetadataCatalog.get("data_detection_val").thing_classes = ["ship","storage tank","baseball diamond","tennis court","plane"] 

In [ ]:
'''
# Visualize some samples using Visualizer to make sure that the function works correctly
# TODO: approx 5 lines
'''
data = DatasetCatalog.get("data_detection_train")

for d in random.sample(data, 3):
    img = cv2.imread(d["file_name"])
    visualizer = Visualizer(img[:, :, ::-1], metadata=MetadataCatalog.get("data_detection_train"), scale=1.2)
    out = visualizer.draw_dataset_dict(d)
    cv2.imwrite(OUTPUT_DIR+'/output_'+str(d['image_id'])+'.jpg', out.get_image()[:, :, ::-1])


### Set Configs

In [ ]:
'''
# Set the configs for the detection part in here.
# TODO: approx 15 lines
'''
cfg = get_cfg()
cfg.OUTPUT_DIR = "{}/output/".format(BASE_DIR)
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml")
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 500
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512


### Training

In [ ]:
'''
# Create a DefaultTrainer using the above config and train the model
# TODO: approx 5 lines
'''
from detectron2.utils.events import TensorboardXWriter

class MyTrainer(DefaultTrainer):
    def __init__(self, cfg):
            super().__init__(cfg)
            self.cfg = cfg

    
    def build_writers(cls):
        # Use the default writers and add TensorboardXWriter
        writers = super().build_writers()
        writers.append(TensorboardXWriter(cls.cfg.OUTPUT_DIR))
        return writers
    
    
cfg.DATASETS.TRAIN = ("data_detection_train",)
trainer = MyTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()


In [ ]:
%reload_ext tensorboard
%load_ext tensorboard
%tensorboard --logdir ./CMPT_CV_lab4/output/

### Evaluation and Visualization

In [ ]:

cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final_base.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.6
predictor = DefaultPredictor(cfg)


In [ ]:
'''
# Visualize the output for 3 random test samples
# TODO: approx 10 lines
'''

data = DatasetCatalog.get("data_detection_test")
for d in random.sample(data, 3):   
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)
    v = Visualizer(im[:, :, ::-1], MetadataCatalog.get("data_detection_test"), scale=1.2)
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    cv2.imwrite(OUTPUT_DIR+'/test_visualize_'+str(d["image_id"])+'.jpg', out.get_image()[:, :, ::-1])



In [ ]:
'''
# Use COCOEvaluator and build_detection_train_loader
# You can save the output predictions using inference_on_dataset
# TODO: approx 5 lines
'''

evaluator = COCOEvaluator("data_detection_val", output_dir=OUTPUT_DIR)
val_loader = build_detection_test_loader(cfg, "data_detection_val")

results = inference_on_dataset(predictor.model, val_loader, evaluator)


### Improvements

For this part, you can bring any improvement which you have by adding new input parameters to the previous functions or defining new functions and variables.

In [ ]:
#Make a new dataloader that splits the images

import copy
from detectron2.data import DatasetMapper
from detectron2.data import detection_utils as utils


def split_image_into_blocks(image, block_size, overlap):
    """Split an image into blocks with overlap."""
    h, w, _ = image.shape
    stride = block_size - overlap
    blocks = []
    block_coords = []

    for y in range(0, h - block_size + 1, stride):
        for x in range(0, w - block_size + 1, stride):
            block = image[y:y + block_size, x:x + block_size]
            blocks.append(block)
            block_coords.append((x, y))  # Top-left coordinates of the block

    return blocks, block_coords


def adjust_bounding_boxes(block_coords, block_size, annotations):
    """Adjust bounding boxes for a specific block."""
    x_offset, y_offset = block_coords
    adjusted_annotations = []

    for ann in annotations:
        x1, y1, w, h = ann["bbox"]
        x2, y2 = x1 + w, y1 + h

        #get box in terms of blocks coords
        x1_adj = max(x1 - x_offset, 0)
        y1_adj = max(y1 - y_offset, 0)
        x2_adj = min(x2 - x_offset, block_size)
        y2_adj = min(y2 - y_offset, block_size)

        #Check that the box is inside the block 
        if x1_adj < x2_adj and y1_adj < y2_adj:  
            ann["bbox"] = [x1_adj, y1_adj, x2_adj - x1_adj, y2_adj - y1_adj]
            adjusted_annotations.append(ann)

    return adjusted_annotations


class blocksMapper(DatasetMapper):
    def __init__(self, cfg, block_size, overlap, is_train=True):
        super().__init__(cfg, is_train=is_train)
        self.block_size = block_size
        self.overlap = overlap

    def __call__(self,dataset_dict):
        # Load the image
        image = utils.read_image(dataset_dict["file_name"], format=self.image_format)

        # Split the image into blocks
        blocks, block_coords = split_image_into_blocks(image, self.block_size, self.overlap)

        for block, coords in zip(blocks, block_coords):
            # Copy the original dataset_dict for this block
            block_dict = copy.deepcopy(dataset_dict)
            
            # Adjust bounding boxes for the block
            block_dict["instances"] = utils.annotations_to_instances(adjust_bounding_boxes(coords, self.block_size, block_dict["annotations"]), block.shape[:2])
            
            # Update block-specific fields
            block_dict["image"] = torch.as_tensor(block.transpose(2, 0, 1).copy(), dtype=torch.float32)  # Copy to avoid negative strides
            block_dict["height"], block_dict["width"] = self.block_size, self.block_size

            if block_dict["instances"]:
                yield block_dict

class BlocksDataset(IterableDataset):
    def __init__(self, dataset, mapper):
        self.dataset = dataset
        self.mapper = mapper

    def __iter__(self):
        while True:
            for dataset_dict in self.dataset:
                for block_dict in self.mapper(dataset_dict):
                    yield block_dict

def dict_collate(batch):
    return batch


def block_data_loader(dataset, mapper, batch_size):
    blocks_dataset = BlocksDataset(dataset, mapper)
    return DataLoader(blocks_dataset, batch_size=batch_size, collate_fn=dict_collate)


In [ ]:
block_size = 1024
overlap = 100
from detectron2.config import CfgNode
block_cfg = get_cfg()
block_cfg.OUTPUT_DIR = "{}/output/".format(BASE_DIR)
block_cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml"))
block_cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml")
block_cfg.SOLVER.IMS_PER_BATCH = 2
block_cfg.SOLVER.BASE_LR = 0.00025
block_cfg.SOLVER.MAX_ITER = 400 #400 Seems to be good for these settings
block_cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
block_cfg.DATASETS.TRAIN = ("data_detection_train",)
block_cfg.DATASETS.TEST = ()
block_cfg.SPLIT = CfgNode()
block_cfg.SPLIT.BLOCK_SIZE = block_size
block_cfg.SPLIT.OVERLAP = overlap

In [ ]:

class MyBlockTrainer(DefaultTrainer):
    def __init__(self, cfg):
            super().__init__(cfg)
            self.cfg = cfg

    @classmethod
    def build_train_loader(cls, cfg):
        mapper = blocksMapper(cfg, block_size=cfg.SPLIT.BLOCK_SIZE, overlap=cfg.SPLIT.OVERLAP)
        return block_data_loader(DatasetCatalog.get(cfg.DATASETS.TRAIN[0]), mapper, batch_size=cfg.SOLVER.IMS_PER_BATCH)
    
    def build_writers(cls):
        writers = super().build_writers()
        writers.append(TensorboardXWriter(cls.cfg.OUTPUT_DIR))
        return writers



blockTrainer = MyBlockTrainer(block_cfg)
blockTrainer.resume_or_load(resume=False)
blockTrainer.train()


In [ ]:
%reload_ext tensorboard
%load_ext tensorboard
%tensorboard --logdir ./CMPT_CV_lab4/output/

In [ ]:
# Prediction Visualization
from detectron2.structures import Instances, Boxes

def combine_block_predictions(block_predictions, block_coords, original_shape, nms_threshold=0.20):
    full_boxes = []
    full_scores = []
    full_classes = []

    for block_pred, (x_offset, y_offset) in zip(block_predictions, block_coords):
        if "instances" not in block_pred:
            continue

        instances = block_pred["instances"]
        boxes = instances.pred_boxes.tensor.to("cpu").numpy()
        scores = instances.scores.to("cpu").numpy()
        classes = instances.pred_classes.to("cpu").numpy()

        # Adjust boxes to original image coordinates
        boxes[:, 0] += x_offset
        boxes[:, 1] += y_offset
        boxes[:, 2] += x_offset
        boxes[:, 3] += y_offset

        full_boxes.append(boxes)
        full_scores.append(scores)
        full_classes.append(classes)
    
    if len(full_boxes) == 0:
        combined_instances = Instances(original_shape)
        combined_instances.pred_boxes = Boxes(torch.tensor(np.array([])))
        combined_instances.scores = torch.tensor(np.array([]))
        combined_instances.pred_classes = torch.tensor(np.array([]))
        return {"instances":combined_instances}

    # Concatenate predictions from all blocks
    full_boxes = np.vstack(full_boxes)
    full_scores = np.hstack(full_scores)
    full_classes = np.hstack(full_classes)

    # Apply Non-Maximum Suppression (NMS)
    # print("DOING NMS")
    keep = torch.ops.torchvision.nms(torch.tensor(full_boxes), torch.tensor(full_scores), nms_threshold)
    combined_boxes = full_boxes[keep.numpy()]
    combined_scores = full_scores[keep.numpy()]
    combined_classes = full_classes[keep.numpy()]

    # Create a single Instances object for the combined predictions
    combined_instances = Instances(original_shape)
    combined_instances.pred_boxes = Boxes(torch.tensor(combined_boxes))
    combined_instances.scores = torch.tensor(combined_scores)
    combined_instances.pred_classes = torch.tensor(combined_classes)

    return {"instances":combined_instances}


def sliced_prediction(predictor , image, block_size, overlap):
    blocks, block_coords = split_image_into_blocks(image, block_size, overlap)
    # print("Done Splitting")
    block_predictions = []
    for block in blocks:
        predictions = predictor(block)
        block_predictions.append(predictions)
    # print("Combining Blocks")
    combined_predictions = combine_block_predictions(block_predictions, block_coords, image.shape[:2])
    return combined_predictions


class slicePredictor(DefaultPredictor):
    def __init__(self, cfg, block_size, overlap):
        # Initialize the config and the model
        super().__init__(cfg)
        self.cfg = cfg
        self.block_size = block_size
        self.overlap = overlap
    
    def __call__(self, original_image):
        self.model.eval()
        outputs = []
        if type(original_image) == list:
            for sample in original_image:
                image = utils.read_image(sample["file_name"], format="BGR")
                outputs.append(sliced_prediction(super().__call__ ,image, self.block_size, self.overlap))
        else:
            return sliced_prediction(super().__call__ ,original_image, self.block_size, self.overlap)
        return outputs



In [ ]:
block_cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
block_cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.8
# cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.8

In [ ]:

data = DatasetCatalog.get("data_detection_test")

for d in random.sample(data, 3): 
    im = cv2.imread(d["file_name"])
    print(d["image_id"])

    slice_predictor = slicePredictor(block_cfg,700, 150)#Seems like good distance, My model was definetly trained way to close.
    outputs = slice_predictor(im)

    # Visualize combined predictions
    visualizer = Visualizer(im[:, :, ::-1], metadata=MetadataCatalog.get("data_detection_test"))
    out = visualizer.draw_instance_predictions(outputs["instances"].to("cpu"))
    cv2.imwrite(OUTPUT_DIR+'/test_visualize_block_'+str(d["image_id"])+'.jpg', out.get_image()[:, :, ::-1])

In [ ]:
evaluator = COCOEvaluator("data_detection_val", output_dir=OUTPUT_DIR)
val_loader = build_detection_test_loader(block_cfg, "data_detection_val")
slice_predictor = slicePredictor(block_cfg,700, 150)
results = inference_on_dataset(slice_predictor, val_loader, evaluator)

## Part 2: Semantic Segmentation

### Data Loader

In [ ]:
'''
# Write a function that returns the cropped image and corresponding mask regarding the target bounding box
# idx is the index of the target bbox in the data
# high-resolution image could be passed or could be load from data['file_name']
# You can use the mask attribute of detectron2.utils.visualizer.GenericMask
#     to convert the segmentation annotations to binary masks
# TODO: approx 10 lines
'''
from detectron2.utils.visualizer import GenericMask
def get_instance_sample(data, idx, img=None):
  if img is None:
    img = cv2.imread(data["file_name"])
      
  bbox = data["annotations"][idx]["bbox"]
  segmenation = data["annotations"][idx]["segmentation"]
  mask = GenericMask(segmenation, img.shape[0], img.shape[1]).mask

  x1, y1, w, h = bbox
  x1, y1, x2, y2 = int(x1), int(y1), int(x1 + w), int(y1 + h)
  
  obj_img = img[y1:y2, x1:x2]
  obj_mask = mask[y1:y2, x1:x2]

  return obj_img, obj_mask

In [ ]:
'''
# We have provided a template data loader for your segmentation training
# You need to complete the __getitem__() function before running the code
# You may also need to add data augmentation or normalization in here
'''

class PlaneDataset(Dataset):
  def __init__(self, set_name, data_list):
      self.transforms = transforms.Compose([
          transforms.ToTensor(), # Converting the image to tensor and change the image format (Channels-Last => Channels-First)
      ])
      self.set_name = set_name
      self.data = data_list
      self.instance_map = []
      for i, d in enumerate(self.data):
        for j in range(len(d['annotations'])):
          self.instance_map.append([i,j])

  '''
  # you can change the value of length to a small number like 10 for debugging of your training procedure and overfeating
  # make sure to use the correct length for the final training
  '''
  def __len__(self):
      return len(self.instance_map)

  def numpy_to_tensor(self, img, mask):
    if self.transforms is not None:
        img = self.transforms(img)
    img = torch.tensor(img, dtype=torch.float)
    mask = torch.tensor(mask, dtype=torch.float)
    return img, mask

  '''
  # Complete this part by using get_instance_sample function
  # make sure to resize the img and mask to a fixed size (for example 128*128)
  # you can use "interpolate" function of pytorch or "numpy.resize"
  # TODO: 5 lines
  '''
  def __getitem__(self, idx):
    if torch.is_tensor(idx):
        idx = idx.tolist()
    idx = self.instance_map[idx]
    data = self.data[idx[0]]

    obj_img, obj_mask = get_instance_sample(data,idx[1])
    img = F.interpolate(torch.tensor(obj_img).unsqueeze(0).permute(0, 3, 1, 2), size=(128, 128), mode='bilinear', align_corners=False).squeeze(0)
    mask = F.interpolate(torch.tensor(obj_mask).unsqueeze(0).unsqueeze(0), size=(128, 128), mode='nearest').squeeze(0)
    img = img/255
    mask = 1.0*mask
    return img, mask

def get_plane_dataset(set_name='train', batch_size=2):
    my_data_list = DatasetCatalog.get("data_detection_{}".format(set_name))
    dataset = PlaneDataset(set_name, my_data_list)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=4,
                                              pin_memory=True, shuffle=True)
    return loader, dataset

### Network

In [ ]:
'''
# convolution module as a template layer consists of conv2d layer, batch normalization, and relu activation
'''
class conv(nn.Module):
    def __init__(self, in_ch, out_ch, activation=True):
        super(conv, self).__init__()
        if(activation):
          self.layer = nn.Sequential(
             nn.Conv2d(in_ch, out_ch, 3, padding=1),
             nn.BatchNorm2d(out_ch),
             nn.ReLU(inplace=True)
          )
        else:
          self.layer = nn.Sequential(
             nn.Conv2d(in_ch, out_ch, 3, padding=1)
             )

    def forward(self, x):
        x = self.layer(x)
        return x

'''
# downsampling module equal to a conv module followed by a max-pool layer
'''
class down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(down, self).__init__()
        self.layer = nn.Sequential(
            conv(in_ch, out_ch),
            nn.MaxPool2d(2)
            )

    def forward(self, x):
        x = self.layer(x)
        return x

'''
# upsampling module equal to a upsample function followed by a conv module
'''
class up(nn.Module):
    def __init__(self, in_ch, out_ch, bilinear=False):
        super(up, self).__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)

        self.conv = conv(in_ch, out_ch)

    def forward(self, x):
        y = self.up(x)
        y = self.conv(y)
        return y

'''
# the main model which you need to complete by using above modules.
# you can also modify the above modules in order to improve your results.
'''
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()

        # Encoder
        self.input_conv1 = conv(3, 16)
        self.input_conv2 = conv(16, 16)
        self.down1 = down(16, 32)
        
        self.input_conv3 = conv(32, 32)
        self.input_conv4 = conv(32, 32)

        # Decoder
        self.up1 = up(32, 16, True)
        self.output_conv2 = conv(32, 16, False)
        self.output_conv1 = conv(16, 1, False)


    def forward(self, input):
        #Encoder
        y = self.input_conv1(input)
        y = self.input_conv2(y)
        skip1 = y
        y = self.down1(y)

        y = self.input_conv3(y)
        y = self.input_conv4(y)
    
      
        #Decoder
        y = self.up1(y)
        
        y = F.interpolate(y, size=skip1.shape[2:], mode='bilinear', align_corners=True)
        y = torch.cat([y, skip1], dim=1)
        y = self.output_conv2(y)
        
        output = self.output_conv1(y)
        return output

### Training

In [ ]:
'''
# The following is a basic training procedure to train the network
# You need to update the code to get the best performance
# TODO: approx ? lines
'''

import matplotlib.pyplot as plt

# Set the hyperparameters
num_epochs = 10
batch_size = 4
learning_rate = 0.01
weight_decay = 1e-5

model = MyModel() # initialize the model
model = model.cuda() # move the model to GPU
loader, _ = get_plane_dataset('train', batch_size) # initialize data_loader
crit = nn.BCEWithLogitsLoss() # Define the loss function
optim = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay) # Initialize the optimizer as SGD

# start the training procedure
lossValues = []
for epoch in range(num_epochs):
  total_loss = 0
  for (img, mask) in tqdm(loader):
    img = torch.tensor(img, device=torch.device('cuda'), requires_grad = True)
    mask = torch.tensor(mask, device=torch.device('cuda'), requires_grad = True)
    pred = model(img)
    loss = crit(pred, mask)
    optim.zero_grad()
    loss.backward()
    optim.step()
    total_loss += loss.cpu().data
    lossValues.append(loss.cpu().data)
  print("Epoch: {}, Loss: {}".format(epoch, total_loss/len(loader)))
  torch.save(model.state_dict(), '{}/output/{}_segmentation_model.pth'.format(BASE_DIR, epoch))

'''
# Saving the final model
'''
torch.save(model.state_dict(), '{}/output/final_segmentation_model.pth'.format(BASE_DIR))

plt.plot(lossValues)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()



### Evaluation and Visualization

In [ ]:
model = MyModel().cuda()
model.load_state_dict(torch.load('{}/output/final_segmentation_model_main.pth'.format(BASE_DIR)))
model = model.eval()

In [ ]:
'''
# Before starting the evaluation, you need to set the model mode to eval
# You may load the trained model again, in case if you want to continue your code later
# TODO: approx 15 lines
'''
batch_size = 8
model = MyModel().cuda()
model.load_state_dict(torch.load('{}/output/final_segmentation_model.pth'.format(BASE_DIR)))
model = model.eval() # chaning the model to evaluation mode will fix the bachnorm layers
loader, dataset = get_plane_dataset('train', batch_size)

total_iou = 0
total_images = 0
for (img, mask) in tqdm(loader):
  with torch.no_grad():
    img = img.cuda()
    mask = mask.cuda()
    mask = torch.squeeze(mask,1)
    

    pred = model(img)
    pred = pred.squeeze(1)

    pred = torch.sigmoid(pred)  # Make predictions probabilities
    pred_binary = (pred > 0.5).float()
    intersection = (pred * mask).sum(dim=(1, 2))
    
    union = ((pred + mask) > 0).sum(dim=(1, 2))
    
    iou = (intersection + 1e-6) / (union + 1e-6)
    
    
    total_iou += iou.sum().item()
    total_images += img.shape[0]

    mean_iou = total_iou / total_images
    '''
    ## Complete the code by obtaining the IoU for each img and print the final Mean IoU
    '''


print("\n #images: {}, Mean IoU: {}".format(total_images, mean_iou))


In [ ]:
'''
# Visualize 3 sample outputs
# TODO: approx 5 lines
'''
model = model.eval()

for i in [1,2,3]:
    img = cv2.imread("./CMPT_CV_lab4/data/PlaneMaskTest"+str(i)+".png")
    
    img = img.astype(np.float32) / 255.0 
    
    
    img_tensor = torch.from_numpy(img).permute(2, 0, 1)
    img_tensor = img_tensor.unsqueeze(0)

    with torch.no_grad():
        pred = model(img_tensor.cuda())
    pred = torch.sigmoid(pred)
    pred_binary = (pred > 0.5).float()
    pred_mask = pred_binary.squeeze(0).cpu().numpy()

    pred_mask = (pred_mask.astype(np.float32) / pred_mask.max()) * 255
    pred_mask = pred_mask.astype(np.uint8)
    pred_mask = pred_mask.reshape((pred_mask.shape[1:]))

    img = img_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() 
    img = (img * 255).astype(np.uint8)
    

    fig, axes = plt.subplots(1, 2, figsize = (20,20))

    axes[0].imshow(img)
    axes[0].set_title("Image")
    axes[0].axis("off") 

    axes[1].imshow(pred_mask, cmap="gray")
    axes[1].set_title("Prediction")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


## Part 3: Instance Segmentation

In this part, you need to obtain the instance segmentation results for the test data by using the trained segmentation model in the previous part and the detection model in Part 1.

### Get Prediction

In [ ]:
'''
# Define a new function to obtain the prediction mask by passing a sample data
# For this part, you need to use all the previous parts (predictor, get_instance_sample, data preprocessings, etc)
# It is better to keep everything (as well as the output of this funcion) on gpu as tensors to speed up the operations.
# pred_mask is the instance segmentation result and should have different values for different planes.
# TODO: approx 35 lines
'''

def get_prediction_mask(data):

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  #Get the image
  img = cv2.imread(data["file_name"])

  #Predict the bound boxes using Predictor
  slicepred = slicePredictor(block_cfg, block_size=700, overlap=150)
  outputs = slicepred(img)

  #Get the predicted boxes
  img_tensor = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
  
  fields= outputs['instances'].get_fields()
  boxes = fields['pred_boxes'].tensor

  #Setup the prediction mask
  h, w = img_tensor.shape[2:]
  pred_mask = torch.zeros((h, w),device=device, dtype=torch.int32)

  for box in boxes:
    x1, y1, x2, y2 = box.int()

    cropped_img = img_tensor[:,:, y1:y2, x1:x2]
    
    with torch.no_grad():
      plane_mask = model(cropped_img.cuda())
    plane_mask = torch.sigmoid(plane_mask)
    plane_mask= (plane_mask > 0.5)

    pred_mask[y1:y2, x1:x2] = torch.where(plane_mask, 1, pred_mask[y1:y2, x1:x2])

  if data["annotations"]:
    gt_mask = torch.zeros((h, w),device=device, dtype=torch.int32)
    annotations = data["annotations"]
    for i in range(len(annotations)):
      _, gt_plane_mask = get_instance_sample(data,i,)
      bbox = data["annotations"][i]["bbox"]
      x1, y1, w, h = bbox
      x1, y1, x2, y2 = int(x1), int(y1), int(x1 + w), int(y1 + h)
      tgt = torch.from_numpy(gt_plane_mask).to(device)
      gt_mask[y1:y2, x1:x2] = torch.where(tgt == 1, 1, gt_mask[y1:y2, x1:x2])
  else:
    gt_mask = torch.zeros((h, w),device=device, dtype=torch.int32)

  return img_tensor, gt_mask, pred_mask 


### Visualization and Submission

In [ ]:
data = DatasetCatalog.get("data_detection_train")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

'''
# Visualise the output prediction as well as the GT Mask and Input image for a sample input
# TODO: approx 10 lines
'''
for d in random.sample(data, 3): 
    img, gt_mask, pred_mask = get_prediction_mask(d)

    pred_mask = pred_mask.cpu().numpy()

    pred_mask = (pred_mask.astype(np.float32) / pred_mask.max()) * 255
    pred_mask = pred_mask.astype(np.uint8)

    gt_mask = gt_mask.cpu().numpy()

    gt_mask = (gt_mask.astype(np.float32) / gt_mask.max()) * 255
    gt_mask = gt_mask.astype(np.uint8)

    img = img.squeeze(0).permute(1, 2, 0).cpu().numpy()  # [H, W, C]
    img = (img * 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize = (20,20))

    axes[0].imshow(img)
    axes[0].set_title("Image")
    axes[0].axis("off") 

    axes[1].imshow(gt_mask, cmap="gray")
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Prediction")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()



In [ ]:
'''
# ref: https://www.kaggle.com/rakhlin/fast-run-length-encoding-python
# https://www.kaggle.com/c/airbus-ship-detection/overview/evaluation
'''
def rle_encoding(x):
    '''
    x: pytorch tensor on gpu, 1 - mask, 0 - background
    Returns run length as list
    '''
    dots = torch.where(torch.flatten(x.long())==1)[0]
    if(len(dots)==0):
      return []
    inds = torch.where(dots[1:]!=dots[:-1]+1)[0]+1
    inds = torch.cat((torch.tensor([0], device=torch.device('cuda'), dtype=torch.long), inds))
    tmpdots = dots[inds]
    inds = torch.cat((inds, torch.tensor([len(dots)], device=torch.device('cuda'))))
    inds = inds[1:] - inds[:-1]
    runs = torch.cat((tmpdots, inds)).reshape((2,-1))
    runs = torch.flatten(torch.transpose(runs, 0, 1)).cpu().data.numpy()
    return ' '.join([str(i) for i in runs])

In [ ]:
'''
# You need to upload the csv file on kaggle
# The speed of your code in the previous parts highly affects the running time of this part
'''

preddic = {"ImageId": [], "EncodedPixels": []}

'''
# Writing the predictions of the training set
'''
my_data_list = DatasetCatalog.get("data_detection_{}".format('train'))
for i in tqdm(range(len(my_data_list)), position=0, leave=True):
  sample = my_data_list[i]
  sample['image_id'] = sample['file_name'].split("/")[-1][:-4]
  img, true_mask, pred_mask = get_prediction_mask(sample)
  inds = torch.unique(pred_mask)
  if(len(inds)==1):
    preddic['ImageId'].append(sample['image_id'])
    preddic['EncodedPixels'].append([])
  else:
    for index in inds:
      if(index == 0):
        continue
      tmp_mask = (pred_mask==index)
      encPix = rle_encoding(tmp_mask)
      preddic['ImageId'].append(sample['image_id'])
      preddic['EncodedPixels'].append(encPix)

'''
# Writing the predictions of the val set
'''

my_data_list = DatasetCatalog.get("data_detection_{}".format('val'))
for i in tqdm(range(len(my_data_list)), position=0, leave=True):
  sample = my_data_list[i]
  sample['image_id'] = sample['file_name'].split("/")[-1][:-4]
  img, true_mask, pred_mask = get_prediction_mask(sample)
  inds = torch.unique(pred_mask)
  if(len(inds)==1):
    preddic['ImageId'].append(sample['image_id'])
    preddic['EncodedPixels'].append([])
  else:
    for index in inds:
      if(index == 0):
        continue
      tmp_mask = (pred_mask==index)
      encPix = rle_encoding(tmp_mask)
      preddic['ImageId'].append(sample['image_id'])
      preddic['EncodedPixels'].append(encPix)

'''
# Writing the predictions of the test set
'''

my_data_list = DatasetCatalog.get("data_detection_{}".format('test'))
for i in tqdm(range(len(my_data_list)), position=0, leave=True):
  sample = my_data_list[i]
  sample['image_id'] = sample['file_name'].split("/")[-1][:-4]
  img, true_mask, pred_mask = get_prediction_mask(sample)
  inds = torch.unique(pred_mask)
  if(len(inds)==1):
    preddic['ImageId'].append(sample['image_id'])
    preddic['EncodedPixels'].append([])
  else:
    for j, index in enumerate(inds):
      if(index == 0):
        continue
      tmp_mask = (pred_mask==index).double()
      encPix = rle_encoding(tmp_mask)
      preddic['ImageId'].append(sample['image_id'])
      preddic['EncodedPixels'].append(encPix)

pred_file = open("{}/pred.csv".format(BASE_DIR), 'w')
pd.DataFrame(preddic).to_csv(pred_file, index=False)
pred_file.close()


## Part 4: Mask R-CNN

For this part you need to follow a same procedure to part 2 with the configs of Mask R-CNN, other parts are generally the same as part 2.

### Data Loader

In [ ]:

import copy
from detectron2.data import DatasetMapper
from detectron2.data import detection_utils as utils


def split_image_into_blocks(image, block_size, overlap):
    """Split an image into blocks with overlap."""
    h, w, _ = image.shape
    stride = block_size - overlap
    blocks = []
    block_coords = []

    for y in range(0, h - block_size + 1, stride):
        for x in range(0, w - block_size + 1, stride):
            block = image[y:y + block_size, x:x + block_size]
            blocks.append(block)
            block_coords.append((x, y))  # Top-left coordinates of the block

    return blocks, block_coords


def adjust_segmenations(block_coords, block_size, annotations):
    """Adjust bounding boxes for a specific block."""
    x_offset, y_offset = block_coords
    adjusted_annotations = []
    
    for ann in annotations:
        newsegs = []
        for seg in ann["segmentation"]:
         # This is a long list of x,y points
            points = np.array(seg).reshape(-1, 2)
            points[:,0] -= x_offset
            points[:,1] -= y_offset

            valid_x = (points[:, 0] >= 0) & (points[:, 0] <= block_size)
            valid_y = (points[:, 1] >= 0) & (points[:, 1] <= block_size)
            valid = valid_x & valid_y
            valid_points = points[valid]
            #Check that the box is inside the block 
            if len(valid_points) > 2:
                newsegs.append(valid_points.flatten().tolist())
        
        if newsegs:
            adjusted_ann = copy.deepcopy(ann)
            adjusted_ann["segmentation"] = newsegs
            adjusted_annotations.append(adjusted_ann)

    return adjusted_annotations


class blocksMapper(DatasetMapper):
    def __init__(self, cfg, block_size, overlap, is_train=True):
        super().__init__(cfg, is_train=is_train)
        self.block_size = block_size
        self.overlap = overlap

    def __call__(self,dataset_dict):
        # Load the image
        image = utils.read_image(dataset_dict["file_name"], format=self.image_format)

        # Split the image into blocks
        blocks, block_coords = split_image_into_blocks(image, self.block_size, self.overlap)

        for block, coords in zip(blocks, block_coords):
            block_dict = copy.deepcopy(dataset_dict)
            
            block_dict["instances"] = utils.annotations_to_instances(adjust_segmenations(coords, self.block_size, block_dict["annotations"]), block.shape[:2])
            print(block_dict)
            
            block_dict["image"] = torch.as_tensor(block.transpose(2, 0, 1).copy(), dtype=torch.float32)  
            block_dict["height"], block_dict["width"] = self.block_size, self.block_size

            if block_dict["instances"]:
                yield block_dict

class BlocksDataset(IterableDataset):
    def __init__(self, dataset, mapper):
        self.dataset = dataset
        self.mapper = mapper

    def __iter__(self):
        while True:
            for dataset_dict in self.dataset:
                for block_dict in self.mapper(dataset_dict):
                    yield block_dict

def dict_collate(batch):
    return batch


def block_data_loader(dataset, mapper, batch_size):
    blocks_dataset = BlocksDataset(dataset, mapper)
    return DataLoader(blocks_dataset, batch_size=batch_size, collate_fn=dict_collate)

### Network

In [ ]:
mask_cfg = get_cfg()
mask_cfg.OUTPUT_DIR = "{}/output/".format(BASE_DIR)
mask_cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
mask_cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
mask_cfg.SOLVER.IMS_PER_BATCH = 2
mask_cfg.SOLVER.BASE_LR = 0.00025
mask_cfg.SOLVER.MAX_ITER = 500
mask_cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
mask_cfg.SPLIT = CfgNode()
mask_cfg.SPLIT.BLOCK_SIZE = 1024
mask_cfg.SPLIT.OVERLAP = 100
mask_cfg.DATASETS.TRAIN = ("data_detection_train",)
mask_cfg.DATASETS.TEST = ()



### Training

In [ ]:
class MaskTrainer(DefaultTrainer):
    def __init__(self, cfg):
            super().__init__(cfg)
            self.cfg = cfg

    @classmethod
    def build_train_loader(cls, cfg):
        mapper = blocksMapper(cfg, block_size=cfg.SPLIT.BLOCK_SIZE, overlap=cfg.SPLIT.OVERLAP)
        return block_data_loader(DatasetCatalog.get(cfg.DATASETS.TRAIN[0]), mapper, batch_size=cfg.SOLVER.IMS_PER_BATCH)
    
    def build_writers(cls):
        # Use the default writers and add TensorboardXWriter
        writers = super().build_writers()
        writers.append(TensorboardXWriter(cls.cfg.OUTPUT_DIR))
        return writers
    
blockTrainer = MaskTrainer(mask_cfg)
maskT = DefaultTrainer(mask_cfg)
maskT.resume_or_load(resume=False)
maskT.train()

### Evaluation and Visualization

In [ ]:
# Prediction Visualization
from detectron2.structures import Instances, Boxes

def split_image_into_blocks(image, block_size, overlap):
    """Split an image into blocks with overlap."""
    h, w, _ = image.shape
    stride = block_size - overlap
    blocks = []
    block_coords = []

    for y in range(0, h - block_size + 1, stride):
        for x in range(0, w - block_size + 1, stride):
            block = image[y:y + block_size, x:x + block_size]
            blocks.append(block)
            block_coords.append((x, y))  # Top-left coordinates of the block

    return blocks, block_coords

def combine_block_predictions(block_predictions, block_coords, original_shape, nms_threshold=0.20):
   
    full_masks = []
    full_scores = []
    full_classes = []

    for block_pred, (x_offset, y_offset) in zip(block_predictions, block_coords):
        if "instances" not in block_pred:
            continue

        instances = block_pred["instances"]
        masks = instances.pred_masks.cpu().numpy()  # Shape: (N, H_block, W_block)
        scores = instances.scores.cpu().numpy()
        classes = instances.pred_classes.cpu().numpy()

        # Adjust masks to the original image coordinates
        for mask in masks:
            # Create an empty mask for the original image and paste the block mask at the correct position
            adjusted_mask = np.zeros(original_shape, dtype=np.uint8)
            h, w = mask.shape
            adjusted_mask[y_offset:y_offset+h, x_offset:x_offset+w] = mask
            full_masks.append(adjusted_mask)
        
        full_scores.extend(scores)
        full_classes.extend(classes)

    if len(full_masks) == 0:
        combined_instances = Instances(original_shape)
        combined_instances.pred_masks = torch.tensor(np.array([]))
        combined_instances.scores = torch.tensor(np.array([]))
        combined_instances.pred_classes = torch.tensor(np.array([]))
        return {"instances": combined_instances}

    # Stack all masks into a single array for NMS
    full_masks = np.stack(full_masks)
    full_scores = np.array(full_scores)
    full_classes = np.array(full_classes)

    # Apply mask-based Non-Maximum Suppression (NMS)
    keep_indices = []
    for i in range(len(full_masks)):
        keep = True
        for j in range(len(full_masks)):
            if i != j:
                # Compute intersection and union for IoU
                intersection = np.logical_and(full_masks[i], full_masks[j]).sum()
                union = np.logical_or(full_masks[i], full_masks[j]).sum()
                iou = intersection / union if union > 0 else 0

                if iou > nms_threshold and full_scores[j] > full_scores[i]:
                    keep = False
                    break
        if keep:
            keep_indices.append(i)

    combined_masks = full_masks[keep_indices]
    combined_scores = full_scores[keep_indices]
    combined_classes = full_classes[keep_indices]

    # Create a single Instances object for the combined predictions
    combined_instances = Instances(original_shape)
    combined_instances.pred_masks = torch.tensor(combined_masks, dtype=torch.uint8)
    combined_instances.scores = torch.tensor(combined_scores)
    combined_instances.pred_classes = torch.tensor(combined_classes)

    return {"instances": combined_instances}

def sliced_mask_prediction(predictor , image, block_size, overlap):
    blocks, block_coords = split_image_into_blocks(image, block_size, overlap)
    print("Done Splitting")
    block_predictions = []
    for block in blocks:
        predictions = predictor(block)
        block_predictions.append(predictions)
    print("Combining Blocks")
    combined_predictions = combine_block_predictions(block_predictions, block_coords, image.shape[:2])
    return combined_predictions


class mask_slicePredictor(DefaultPredictor):
    def __init__(self, cfg, block_size, overlap):
        # Initialize the config and the model
        super().__init__(cfg)
        self.cfg = cfg
        self.block_size = block_size
        self.overlap = overlap
    
    def __call__(self, original_image):
        self.model.eval()
        outputs = []
        if type(original_image) == list:
            for sample in original_image:
                image = utils.read_image(sample["file_name"], format="BGR")
                outputs.append(sliced_mask_prediction(super().__call__ ,image, self.block_size, self.overlap))
        else:
            return sliced_mask_prediction(super().__call__ ,original_image, self.block_size, self.overlap)
        return outputs



In [ ]:
testdata = DatasetCatalog.get("data_detection_test")

In [ ]:
mask_cfg.MODEL.WEIGHTS = os.path.join(mask_cfg.OUTPUT_DIR, "model_final.pth")
mask_cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.60

In [ ]:
import matplotlib.pyplot as plt


maskpredictor = DefaultPredictor(mask_cfg)

for d in random.sample(testdata, 3):   
    im = cv2.imread(d["file_name"])
    outputs = maskpredictor(im)
    v = Visualizer(im[:, :, ::-1], MetadataCatalog.get("data_detection_test"), scale=1.2)
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    plt.imshow(out.get_image())
    plt.show()



In [ ]:
import matplotlib.pyplot as plt
mask_cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
mask_cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.60

maskpredictor = mask_slicePredictor(mask_cfg,700,100)

for d in random.sample(testdata, 3):   
    im = cv2.imread(d["file_name"])
    outputs = maskpredictor(im)
    v = Visualizer(im[:, :, ::-1], MetadataCatalog.get("data_detection_test"), scale=1.2)
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    plt.imshow(out.get_image())
    plt.show()


In [ ]:
maskpredictor = mask_slicePredictor(mask_cfg,700,100)
evaluator = COCOEvaluator("data_detection_val", output_dir=OUTPUT_DIR)
val_loader = build_detection_test_loader(mask_cfg, "data_detection_val")

results = inference_on_dataset(maskpredictor, val_loader, evaluator)